### Paso 0: Defición del archivo

Este notebook centraliza el flujo de generación de datos sintéticos del proyecto AndesMarket.

El objetivo es producir una capa `data/raw` con errores controlados de calidad para que el notebook `01_datamart_etl.ipynb` pueda ejecutar el proceso ETL y generar la capa final `data/processed`.

Flujo:

1. Cargar configuración y scripts de generación.
2. Cargar dimensiones maestras curadas.
3. Generar clientes, tiempo y ventas.
4. Validar la versión limpia en memoria.
5. Inyectar ruido controlado.
6. Exportar tablas a `data/raw`.

La lógica principal vive en `scripts/`; este notebook actúa como orquestador reproducible.

### Paso 1: Imports y rutas

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import pandas as pd

# Detecta raíz del proyecto ejecutando desde notebooks/ o desde la raíz.
_cwd = Path.cwd()
PROJECT_ROOT = _cwd if (_cwd / "data").exists() else _cwd.parent

DATA_DIR = PROJECT_ROOT / "data"
DATA_RAW = DATA_DIR / "raw"
DATA_PROCESSED = DATA_DIR / "processed"
DATA_DEVELOPMENT = DATA_DIR / "development"
DATA_TABLAS_MAESTRAS = DATA_DIR / "tablas_maestras"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"

for carpeta in [DATA_RAW, DATA_DEVELOPMENT]:
    carpeta.mkdir(parents=True, exist_ok=True)

# Permite importar módulos desde scripts/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SEP = ";"
ENC = "utf-8"

print("Raíz del proyecto:", PROJECT_ROOT)
print("Entrada maestra:", DATA_TABLAS_MAESTRAS)
print("Salida raw:", DATA_RAW)
print("Scripts:", SCRIPTS_DIR)

### Paso 2: Importar scripts del proyecto

In [ ]:
from scripts.config import (
    SEED,
    TOTAL_CLIENTES,
    FECHA_INICIO_OPERACION,
)

from scripts.generar_clientes import generar_clientes
from scripts.generar_ventas import generar_ventas_y_actualizar_clientes
from scripts.generar_dim_tiempo import generar_dim_tiempo

from scripts.inyectar_ruido import inyectar_ruido, resumen_ruido

print("SEED:", SEED)
print("TOTAL_CLIENTES:", TOTAL_CLIENTES)
print("FECHA_INICIO_OPERACION:", FECHA_INICIO_OPERACION)

### Paso 3: Cargar dimensiones curadas

In [ ]:
def cargar_csv_curado(nombre_archivo: str) -> pd.DataFrame:
    """
    Busca dimensiones maestras en orden:
    1. data/development
    2. data/processed
    3. data
    4. z_archive

    Esto permite migrar gradualmente sin romper el proyecto.
    """
    rutas_candidatas = [
        DATA_DEVELOPMENT / nombre_archivo,
        DATA_PROCESSED / nombre_archivo,
        DATA_DIR / nombre_archivo,
        PROJECT_ROOT / "z_archive" / nombre_archivo,
    ]

    for ruta in rutas_candidatas:
        if ruta.exists():
            print(f"Cargando {nombre_archivo} desde {ruta.relative_to(PROJECT_ROOT)}")
            return pd.read_csv(ruta, sep=SEP, encoding=ENC)

    raise FileNotFoundError(f"No se encontró {nombre_archivo} en rutas candidatas: {rutas_candidatas}")


dim_producto = cargar_csv_curado("dim_producto.csv")
dim_tienda = cargar_csv_curado("dim_tienda.csv")
dim_promocion = cargar_csv_curado("dim_promocion.csv")

print("dim_producto:", dim_producto.shape)
print("dim_tienda:", dim_tienda.shape)
print("dim_promocion:", dim_promocion.shape)

### Paso 4: Generar clientes

In [ ]:
dim_cliente = generar_clientes(total_clientes=TOTAL_CLIENTES, seed=SEED)

print("Clientes generados:", len(dim_cliente))
display(dim_cliente.head())

print("Distribución por segmento:")
display(
    dim_cliente["segmento_programa"]
    .value_counts(normalize=True)
    .rename("proporcion")
    .round(3)
    .to_frame()
)

print("Distribución por región:")
display(
    dim_cliente["region"]
    .value_counts(normalize=True)
    .rename("proporcion")
    .round(3)
    .to_frame()
)

### Paso 5: Generar dimensión tiempo

In [ ]:
dim_tiempo = generar_dim_tiempo()

print("Dimensión tiempo generada.")
print("Días generados:", len(dim_tiempo))
print("Fecha mínima:", dim_tiempo["fecha"].min())
print("Fecha máxima:", dim_tiempo["fecha"].max())
print("Feriados:", dim_tiempo["es_feriado"].sum())

display(dim_tiempo.head())
display(dim_tiempo.tail())

### Paso 6: Generar ventas limpias

In [ ]:
dim_cliente_clean, fact_ventas_clean = generar_ventas_y_actualizar_clientes(
    clientes_base=dim_cliente,
    df_producto=dim_producto,
    df_tienda=dim_tienda,
    df_promocion=dim_promocion,
    total_clientes=TOTAL_CLIENTES,
    seed=SEED,
)

print("Clientes finales:", len(dim_cliente_clean))
print("Tickets:", fact_ventas_clean["id_venta"].nunique())
print("Líneas de venta:", len(fact_ventas_clean))
print("Rango de fechas:", fact_ventas_clean["fecha"].min(), "a", fact_ventas_clean["fecha"].max())

display(fact_ventas_clean.head())

### Paso 7: Armar paquete limpio en memoria

In [ ]:
tablas_clean = {
    "dim_cliente": dim_cliente_clean.copy(),
    "dim_producto": dim_producto.copy(),
    "dim_tienda": dim_tienda.copy(),
    "dim_promocion": dim_promocion.copy(),
    "dim_tiempo": dim_tiempo.copy(),
    "fact_ventas": fact_ventas_clean.copy(),
}

for nombre, df in tablas_clean.items():
    print(f"{nombre}: {len(df):,} filas, {len(df.columns)} columnas")

### Paso 8: Validar data generada

**Permite verificiar que**
- no hay clientes sin venta
- no hay clientes sin fecha_alta
- fecha_alta coincide con primera venta
- no hay duplicados de grano
- no hay FK huérfanas
- las fechas existen en dim_tiempo

In [ ]:
def validar_version_limpia(tablas: dict[str, pd.DataFrame]) -> pd.DataFrame:
    cliente = tablas["dim_cliente"]
    producto = tablas["dim_producto"]
    tienda = tablas["dim_tienda"]
    promocion = tablas["dim_promocion"]
    tiempo = tablas["dim_tiempo"]
    ventas = tablas["fact_ventas"]

    checks = {
        "clientes_generados": len(cliente),
        "productos": len(producto),
        "tiendas": len(tienda),
        "promociones": len(promocion),
        "dias_dim_tiempo": len(tiempo),
        "tickets_venta": ventas["id_venta"].nunique(),
        "lineas_venta": len(ventas),

        "clientes_sin_venta": len(set(cliente["id_cliente"]) - set(ventas["id_cliente"])),
        "duplicados_grano_fact": int(ventas.duplicated(["id_venta", "numero_linea"]).sum()),

        "fk_cliente_huerfanas": len(set(ventas["id_cliente"]) - set(cliente["id_cliente"])),
        "fk_producto_huerfanas": len(set(ventas["id_producto"]) - set(producto["id_producto"])),
        "fk_tienda_huerfanas": len(set(ventas["id_tienda"]) - set(tienda["id_tienda"])),
        "fk_promocion_huerfanas": len(set(ventas["id_promocion"]) - set(promocion["id_promocion"])),
        "fk_fecha_huerfanas": len(set(ventas["fecha"]) - set(tiempo["fecha"])),
    }

    return pd.DataFrame(
        [{"validacion": k, "valor": v} for k, v in checks.items()]
    )


validacion_clean = validar_version_limpia(tablas_clean)
display(validacion_clean)

### Paso 9: Inyectar ruido controlado

In [ ]:
tablas_raw = inyectar_ruido(tablas_clean, seed=SEED)

resumen = resumen_ruido(tablas_raw)
display(pd.DataFrame([resumen]).T.rename(columns={0: "valor"}))

print("Ejemplos de fechas raw:")
print(tablas_raw["fact_ventas"]["fecha"].head(10).tolist())

### Paso 10: Exportar a data/raw

In [ ]:
TABLAS_SALIDA = {
    "dim_cliente": "dim_cliente.csv",
    "dim_producto": "dim_producto.csv",
    "dim_tienda": "dim_tienda.csv",
    "dim_promocion": "dim_promocion.csv",
    "dim_tiempo": "dim_tiempo.csv",
    "fact_ventas": "fact_ventas.csv",
}


def exportar_tablas(tablas: dict[str, pd.DataFrame], carpeta: Path) -> None:
    carpeta.mkdir(parents=True, exist_ok=True)

    for nombre, archivo in TABLAS_SALIDA.items():
        ruta = carpeta / archivo
        tablas[nombre].to_csv(ruta, sep=SEP, index=False, encoding=ENC)
        print(f"Exportado {ruta.relative_to(PROJECT_ROOT)}: {len(tablas[nombre]):,} filas")


exportar_tablas(tablas_raw, DATA_RAW)

print("\nGeneración centralizada terminada.")
print("Siguiente paso: ejecutar 01_datamart_etl.ipynb para limpiar data/raw y generar data/processed.")

### Paso 11: Resumen ejecutivo final

In [ ]:
resumen_final = {
    "clientes_generados": len(tablas_clean["dim_cliente"]),
    "productos": len(tablas_clean["dim_producto"]),
    "tiendas": len(tablas_clean["dim_tienda"]),
    "promociones": len(tablas_clean["dim_promocion"]),
    "dias_dim_tiempo": len(tablas_clean["dim_tiempo"]),
    "tickets_limpios": tablas_clean["fact_ventas"]["id_venta"].nunique(),
    "lineas_limpias": len(tablas_clean["fact_ventas"]),
    "lineas_raw": len(tablas_raw["fact_ventas"]),
    "duplicados_raw_grano": int(tablas_raw["fact_ventas"].duplicated(["id_venta", "numero_linea"]).sum()),
    "nulos_descuento_raw": int(tablas_raw["fact_ventas"]["descuento_pct"].isna().sum()),
}

display(pd.DataFrame([resumen_final]).T.rename(columns={0: "valor"}))